# Problem Statement 6: Data Analytics III – Naïve Bayes Classification
**Objective:** Implement Gaussian Naïve Bayes on the Iris dataset and compute the confusion matrix with all metrics.

**Dataset:** Iris dataset – 150 samples, 4 features, 3 species classes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, ConfusionMatrixDisplay
)

print("Libraries imported!")

## Step 1: Load Iris Dataset

In [ ]:
iris = load_iris()
df   = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df['species_name'].value_counts())
df.head(10)

In [ ]:
# Visualize the dataset
sns.pairplot(df, hue='species_name', vars=iris.feature_names, palette='Set1')
plt.suptitle('Iris Dataset – Pairplot', y=1.02, fontsize=13)
plt.show()

## Step 2: Split Data into Train and Test Sets

In [ ]:
X = df[iris.feature_names]
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Testing set:  {X_test.shape}")

## Step 3: Train Gaussian Naïve Bayes Model
**Theory:** Naïve Bayes assumes all features are independent and uses Bayes' theorem:
P(class|features) ∝ P(features|class) × P(class)

GaussianNB assumes features follow a normal (Gaussian) distribution.

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred = gnb.predict(X_test)

print("Naïve Bayes model trained!")
print("\nFirst 10 Predictions vs Actual:")
result = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
result['Actual_Name']    = result['Actual'].map({0:'setosa', 1:'versicolor', 2:'virginica'})
result['Predicted_Name'] = result['Predicted'].map({0:'setosa', 1:'versicolor', 2:'virginica'})
print(result.head(10))

## Step 4: Confusion Matrix and Metrics

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("\nRows = Actual, Columns = Predicted")
print("Classes: 0=setosa, 1=versicolor, 2=virginica")

In [ ]:
# For multi-class, compute TP, FP, TN, FN per class
print("Per-Class Metrics (One-vs-Rest):")
class_names = ['setosa', 'versicolor', 'virginica']

for i, cls in enumerate(class_names):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - TP - FP - FN
    
    accuracy   = (TP + TN) / (TP + TN + FP + FN)
    error_rate = 1 - accuracy
    precision  = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall     = TP / (TP + FN) if (TP + FN) > 0 else 0
    
    print(f"\n  {cls.upper()}:")
    print(f"    TP={TP}, FP={FP}, TN={TN}, FN={FN}")
    print(f"    Accuracy={accuracy:.4f}, Error Rate={error_rate:.4f}")
    print(f"    Precision={precision:.4f}, Recall={recall:.4f}")

In [ ]:
overall_acc = accuracy_score(y_test, y_pred)
print(f"\nOverall Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Plot Confusion Matrix
plt.figure(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', colorbar=True)
plt.title('Confusion Matrix – Naïve Bayes on Iris Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Class probability predictions
proba = gnb.predict_proba(X_test[:5])
print("Predicted Probabilities for first 5 test samples:")
prob_df = pd.DataFrame(proba, columns=class_names)
prob_df['Predicted'] = y_pred[:5]
prob_df['Actual'] = y_test.values[:5]
print(prob_df)